In [4]:
from typing import NamedTuple, TypedDict, AnyStr, Callable, Any, TypeVar

import pandas as pd
import requests
from pathlib import Path

from narwhals import DataFrame

from tqdm.notebook import tqdm
import zipfile
import librosa
import IPython
from glob import glob
import numpy as np

In [5]:
np.random.seed(0)

In [6]:
# https://stackoverflow.com/a/62113293
def download(url, path: Path):
    resp = requests.get(url, stream=True)
    total = int(resp.headers.get('content-length', 0))
    with open(path, 'wb') as file, tqdm(
            desc=path.name,
            total=total,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
    ) as bar:
        for data in resp.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

In [7]:
download_path = Path("data/dataset.zip")
if not (download_path.exists() and download_path.is_file()):
    download("https://zenodo.org/records/3371780/files/audio_mono-mic.zip?download=1", download_path)

In [8]:
guitar_set_dir = Path("data/GuitarSet")
if not (guitar_set_dir.exists() and guitar_set_dir.is_dir()):
    with zipfile.ZipFile(download_path, "r") as zip_ref:
        for member in tqdm(zip_ref.infolist(), desc="Extracting "):
            zip_ref.extract(member, guitar_set_dir)

In [9]:
# example, sr = librosa.load(guitar_set_dir.joinpath("00_BN1-129-Eb_comp_mic.wav"))
# IPython.display.Audio(data = example, rate=sr)

In [10]:
class Sample(NamedTuple):
    audio: Path
    x: np.ndarray
    sr: int
    sample_start: int
    sample_end: int



class SampleWithFeatures(NamedTuple):
    audio: Path
    x: np.ndarray
    sr: int
    sample_start: int
    sample_end: int
    zero_crossing_rate: float
    spectral_centroid: float
    rms: float

In [11]:
T = TypeVar('T')
def split_into_samples(files: list[AnyStr], sample_length: int, f: Callable[[Sample], T]) -> list[T]:
    dataset_list: list[T] = []
    for file in files:
        try:
            sample, sr = librosa.load(Path(file))
        except Exception as e:
            print(f"Failed to load {file}: {e}")
            continue
        length: float = len(sample) / sr
        sample_no = int(length // sample_length)
        remainder: float = length - sample_length * sample_no

        last_offset:int = 0

        for i in range(sample_no):
            offset: float = np.random.random()*remainder
            remainder -= max(0, offset)
            start_index = last_offset + int(offset*sr)
            end_index = min(int(last_offset + offset*sr + sample_length*sr), len(sample))
            dataset_list.append(f(Sample(audio=Path(file), x=sample, sr=sr, sample_start=start_index, sample_end=end_index)))
            last_offset = end_index
    return dataset_list

In [15]:
files = glob(str(guitar_set_dir)+"/**/*", recursive=True)

['data/GuitarSet/00_SS3-84-Bb_comp_mic.wav',
 'data/GuitarSet/05_BN3-119-G_solo_mic.wav',
 'data/GuitarSet/04_Jazz1-200-B_solo_mic.wav',
 'data/GuitarSet/05_Jazz3-150-C_comp_mic.wav',
 'data/GuitarSet/03_Funk1-97-C_solo_mic.wav',
 'data/GuitarSet/03_BN1-129-Eb_comp_mic.wav',
 'data/GuitarSet/03_Rock2-142-D_comp_mic.wav',
 'data/GuitarSet/03_SS3-98-C_comp_mic.wav',
 'data/GuitarSet/05_Rock3-117-Bb_comp_mic.wav',
 'data/GuitarSet/00_Rock2-142-D_solo_mic.wav',
 'data/GuitarSet/04_BN2-131-B_comp_mic.wav',
 'data/GuitarSet/03_BN1-147-Gb_comp_mic.wav',
 'data/GuitarSet/02_Funk3-98-A_solo_mic.wav',
 'data/GuitarSet/05_Jazz2-187-F#_solo_mic.wav',
 'data/GuitarSet/05_Funk2-119-G_comp_mic.wav',
 'data/GuitarSet/00_SS3-98-C_solo_mic.wav',
 'data/GuitarSet/00_Jazz1-200-B_solo_mic.wav',
 'data/GuitarSet/03_Funk3-98-A_comp_mic.wav',
 'data/GuitarSet/02_BN1-147-Gb_solo_mic.wav',
 'data/GuitarSet/02_Jazz3-150-C_solo_mic.wav',
 'data/GuitarSet/04_Jazz3-137-Eb_solo_mic.wav',
 'data/GuitarSet/03_Jazz1-20

In [13]:
dataset_5sec: DataFrame = pd.DataFrame(split_into_samples(files, 5, lambda x: x))
display(dataset_5sec[dataset_5sec["audio"].apply(str)=="data/GuitarSet/00_SS3-84-Bb_comp_mic.wav"])
dataset_10sec: DataFrame = pd.DataFrame(split_into_samples(files, 10, lambda x: x))
display(dataset_10sec[dataset_10sec["audio"].apply(str)=="data/GuitarSet/00_SS3-84-Bb_comp_mic.wav"])

KeyError: 'audio'

In [ ]:
def get_audio(df: DataFrame, i: int)->tuple[np.ndarray, int]:
    row = df.iloc[i]
    x, sr = librosa.load(row.audio)
    return x[row.sample_start:row.sample_end], sr

In [ ]:
x, sr = get_audio(dataset_10sec, 0)
IPython.display.Audio(x, rate=sr)

In [ ]:
x, sr = get_audio(dataset_5sec, 1)
IPython.display.Audio(x, rate=sr)